# Network Modules, Enrichment & Visualization

**Tier 3 — Applied Bioinformatics | Module 28 · Notebook 2**

*Prerequisites: Notebook 1 (PPI Network Construction)*

---

**By the end of this notebook you will be able to:**
1. Apply Louvain community detection to find functional network modules
2. Run GO enrichment on each detected module
3. Export networks to Cytoscape format for interactive exploration
4. Build protein co-expression networks from RNA-seq data (WGCNA concept)
5. Integrate network modules with differential expression results



**Key resources:**
- [Cytoscape documentation](https://cytoscape.org/documentation_users.html)
- [py4cytoscape documentation](https://py4cytoscape.readthedocs.io/)
- [NetworkX community detection](https://networkx.org/documentation/stable/reference/algorithms/community.html)

---

[< Previous: PPI Network Construction & Analysis](01_ppi_networks.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Gene Regulatory Network Inference >](03_gene_regulatory_networks.ipynb)

---

## 1. Community Detection in Biological Networks

Biological networks are organized into **modules** (communities) — groups of densely interconnected nodes that are sparsely connected to the rest of the network. These modules often correspond to biological functions or complexes.

### Why modules matter
- **Functional coherence**: nodes in the same module tend to share biological function
- **Robustness**: modular organization prevents cascading failures
- **Drug targeting**: disrupt specific modules to affect specific pathways

### Community detection algorithms

| Algorithm | Key idea | Complexity | Notes |
|---|---|---|---|
| **Louvain** | Greedy modularity maximization + refinement | O(n log n) | Fast, widely used |
| **Leiden** | Fixed partition bug in Louvain | O(n log n) | Preferred over Louvain |
| **Girvan-Newman** | Edge betweenness removal | O(m²n) | Slow, interpretable |
| **Spectral clustering** | Eigenvectors of Laplacian | O(n³) | Principled, no resolution limit |
| **Infomap** | Random walk compression | O(m) | Good for directed networks |

### Modularity score Q
Q measures quality of a partition: fraction of edges within modules minus expected fraction under random placement.
- Q > 0.3: meaningful community structure
- Q > 0.5: strong modularity
- Q = 0: no better than random
- Q = 1: perfect (no between-module edges)


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from collections import defaultdict

np.random.seed(42)

# ----- Generate a modular PPI network -----
# 5 biological modules: Cell cycle, DNA repair, Signaling, Metabolism, Immune
module_names = ['Cell_cycle', 'DNA_repair', 'MAPK_signaling', 'Metabolism', 'Immune']
module_sizes = [12, 10, 11, 9, 10]

module_gene_sets = {
    'Cell_cycle':    ['CDK1','CDK2','CDK4','CDK6','CCND1','CCNE1','RB1','E2F1','E2F3','CDKN1A','CDKN2A','CDC25A'],
    'DNA_repair':    ['BRCA1','BRCA2','ATM','ATR','CHEK1','CHEK2','RAD51','FANCD2','PALB2','MLH1'],
    'MAPK_signaling':['KRAS','BRAF','MAP2K1','MAPK1','MAPK3','EGFR','ERBB2','GRB2','SOS1','RAF1','HRAS','PIK3CA'],
    'Metabolism':    ['LDHA','PKM','HK2','G6PD','FASN','ACLY','HMGCR','SLC2A1'],
    'Immune':        ['CD8A','CD274','PDCD1','CTLA4','LAG3','TIM3','TIGIT','FOXP3','IL2','IFNG'],
}
all_genes = [g for gs in module_gene_sets.values() for g in gs]
n_total = len(all_genes)

# Build adjacency matrix with modular structure
# High intra-module connectivity, low inter-module connectivity
G = nx.Graph()
G.add_nodes_from(all_genes)

# Add node module labels
for mod, genes in module_gene_sets.items():
    for g in genes:
        G.nodes[g]['module_true'] = mod

# Intra-module edges (dense)
for mod, genes in module_gene_sets.items():
    n = len(genes)
    for i in range(n):
        for j in range(i+1, n):
            if np.random.random() < 0.55:  # 55% intra-module edge probability
                score = np.random.uniform(500, 999)
                G.add_edge(genes[i], genes[j], weight=score)

# Inter-module edges (sparse, known cross-talk)
cross_talk = [
    ('Cell_cycle', 'DNA_repair', 0.20),  # cell cycle-DNA damage checkpoint
    ('MAPK_signaling', 'Cell_cycle', 0.15),
    ('MAPK_signaling', 'Metabolism', 0.10),
    ('DNA_repair', 'Immune', 0.08),
    ('Immune', 'MAPK_signaling', 0.06),
]
for mod1, mod2, prob in cross_talk:
    for g1 in module_gene_sets[mod1]:
        for g2 in module_gene_sets[mod2]:
            if np.random.random() < prob:
                G.add_edge(g1, g2, weight=np.random.uniform(150, 500))

print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Modules: {module_names}")
print(f"Density: {nx.density(G):.3f}")
print(f"Connected: {nx.is_connected(G)}")


## 2. Louvain Community Detection

The **Louvain algorithm** proceeds in two phases:
1. **Phase 1** (modularity gain): Move each node to the neighbor community that maximizes ΔQ
2. **Phase 2** (community contraction): Aggregate each community into a supernode; repeat

This is repeated until ΔQ = 0 (local maximum of modularity).

### Leiden algorithm (improved Louvain)
Louvain can produce **disconnected communities** (a bug). The Leiden algorithm guarantees:
- Well-connected communities
- Faster convergence
- Better modularity scores

```python
# Louvain (python-louvain package)
import community as community_louvain
partition = community_louvain.best_partition(G, random_state=42)

# Leiden (leidenalg package)
import leidenalg
import igraph as ig
ig_G = ig.Graph.from_networkx(G)
partition = leidenalg.find_partition(ig_G, leidenalg.ModularityVertexPartition, seed=42)
```


In [ ]:
# ----- Community detection: Louvain algorithm -----
# Note: python-louvain package (community module) implements Louvain
# Here we implement a simplified version based on greedy modularity

try:
    import community as community_louvain
    partition_louvain = community_louvain.best_partition(G, random_state=42)
    method = 'Louvain (python-louvain)'
except ImportError:
    # Fallback: greedy modularity maximization from networkx
    communities = nx.community.greedy_modularity_communities(G)
    partition_louvain = {}
    for i, comm in enumerate(communities):
        for node in comm:
            partition_louvain[node] = i
    method = 'Greedy modularity (networkx)'

n_communities = len(set(partition_louvain.values()))
modularity = nx.community.modularity(G, 
    [{n for n, c in partition_louvain.items() if c == i} for i in range(n_communities)])

print(f"Community detection method: {method}")
print(f"Number of communities found: {n_communities}")
print(f"Modularity score Q = {modularity:.4f} (>0.3 is meaningful)")

# Compare detected communities with true modules
print("\nCommunity composition (detected vs true):")
df_comp = pd.DataFrame({'Gene': list(G.nodes()),
                         'Detected': [partition_louvain[n] for n in G.nodes()],
                         'True': [G.nodes[n]['module_true'] for n in G.nodes()]})
print(pd.crosstab(df_comp['True'], df_comp['Detected']))


## 3. Visualizing Network Modules

Good module visualization requires:
1. **Layout that separates modules** (e.g., spring layout with high k)
2. **Color nodes by module membership**
3. **Show inter-module edges** as thin, different color
4. **Export to Cytoscape** for publication-quality figures

### Comparing detected vs. true modules
The **Normalized Mutual Information (NMI)** and **Adjusted Rand Index (ARI)** measure similarity between detected and true partitions:
- NMI = 1: perfect match
- NMI = 0: random match

```python
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
nmi = normalized_mutual_info_score(true_labels, detected_labels)
ari = adjusted_rand_score(true_labels, detected_labels)
```


In [ ]:
# ----- Visualize communities -----

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Network Community Detection', fontsize=13, fontweight='bold')

pos = nx.spring_layout(G, seed=42, k=3.0)

# Panel 1: True modules (ground truth)
mod_color_map = cm.Set1(np.linspace(0, 1, len(module_names)))
mod_to_color = {mod: mod_color_map[i] for i, mod in enumerate(module_names)}
true_colors = [mod_to_color[G.nodes[n]['module_true']] for n in G.nodes()]
nx.draw_networkx(G, pos=pos, ax=axes[0], node_color=true_colors, node_size=150,
                 edge_color='gray', alpha=0.7, width=0.5, font_size=5, arrows=False)
axes[0].set_title('True Module Structure')
handles = [mpatches.Patch(color=mod_to_color[m], label=m.replace('_', ' '))
           for m in module_names]
axes[0].legend(handles=handles, fontsize=7, loc='lower left')
axes[0].axis('off')

# Panel 2: Detected communities
n_detected = len(set(partition_louvain.values()))
det_colors_map = cm.tab10(np.linspace(0, 1, n_detected))
det_colors = [det_colors_map[partition_louvain[n]] for n in G.nodes()]
nx.draw_networkx(G, pos=pos, ax=axes[1], node_color=det_colors, node_size=150,
                 edge_color='gray', alpha=0.7, width=0.5, font_size=5, arrows=False)
axes[1].set_title(f'Detected Communities (Q={modularity:.3f})')
axes[1].axis('off')

# Panel 3: Module size distribution
comm_sizes = pd.Series(partition_louvain).value_counts().sort_index()
axes[2].bar(comm_sizes.index, comm_sizes.values, color=det_colors_map[:len(comm_sizes)])
axes[2].set_xlabel('Community ID')
axes[2].set_ylabel('Number of nodes')
axes[2].set_title('Community Size Distribution')

plt.tight_layout()
plt.show()

print(f"\nCommunity sizes: {dict(comm_sizes)}")
print(f"Modularity Q={modularity:.4f}")
print("Q > 0.3: meaningful community structure")
print("Q > 0.5: strong modularity")


## 4. WGCNA: Weighted Co-expression Network Analysis

**WGCNA** constructs co-expression modules from RNA-seq data (not from PPI databases):

### WGCNA workflow
1. **Pearson correlation matrix** of gene expression
2. **Soft thresholding**: raise adjacency to power β to emphasize strong correlations (scale-free test)
3. **Topological Overlap Measure (TOM)**: connectivity-based adjacency (more robust than simple correlation)
4. **Hierarchical clustering** of 1-TOM distance matrix
5. **Dynamic tree cut**: cut dendrogram to obtain modules
6. **Module eigengene (ME)**: first PC of module expression = summary of module activity

### Key WGCNA outputs
- **Module-trait correlation**: which modules are associated with clinical outcomes?
- **Module membership (kME)**: correlation of each gene with its module eigengene
- **Hub gene**: highest intra-module connectivity (kIM) within a module

### WGCNA vs. PPI network modules
| Aspect | WGCNA | STRING communities |
|---|---|---|
| Data | Expression matrix | Protein interactions |
| Edge | Co-expression r | PPI confidence |
| Interpretation | Co-regulated genes | Physical/functional partners |
| Context-specific | Yes | No |


In [ ]:
# ----- WGCNA concepts: co-expression network modules -----

# WGCNA (Weighted Gene Co-expression Network Analysis) works on expression data
# Step 1: Build adjacency matrix from correlation matrix
# Step 2: Raise to soft threshold power beta (ensures scale-free topology)
# Step 3: TOM (Topological Overlap Measure) = connectivity-based adjacency
# Step 4: Hierarchical clustering + dynamic tree cut → modules
# Step 5: Module eigengene = first PC of module expression

np.random.seed(5)

# Simulate expression data for the network genes
n_samples = 50
n_genes = len(all_genes)
gene_idx = {g: i for i, g in enumerate(all_genes)}

# Expression matrix: genes with module structure
expr_matrix = np.random.randn(n_samples, n_genes)
# Add module signal
module_signals = {mod: np.random.randn(n_samples) for mod in module_names}
for mod, genes in module_gene_sets.items():
    for g in genes:
        expr_matrix[:, gene_idx[g]] += module_signals[mod] * 1.5

# Correlation matrix
from sklearn.preprocessing import StandardScaler
expr_scaled = StandardScaler().fit_transform(expr_matrix)
corr_matrix = np.corrcoef(expr_scaled.T)  # gene x gene

# Module eigengenes
module_eigengenes = {}
from sklearn.decomposition import PCA
for mod, genes in module_gene_sets.items():
    mod_indices = [gene_idx[g] for g in genes]
    mod_expr = expr_scaled[:, mod_indices]
    pca = PCA(n_components=1)
    eigengene = pca.fit_transform(mod_expr).flatten()
    module_eigengenes[mod] = eigengene

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('WGCNA Co-expression Network Concepts', fontsize=12, fontweight='bold')

# 1: Gene correlation heatmap (subset)
# Order genes by module
ordered_genes = [g for mod in module_names for g in module_gene_sets[mod]]
ordered_idx = [gene_idx[g] for g in ordered_genes]
corr_ordered = corr_matrix[np.ix_(ordered_idx, ordered_idx)]

im = axes[0].imshow(corr_ordered, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[0].set_title('Gene-Gene Correlation Matrix\n(ordered by module)')
axes[0].set_xticks([]); axes[0].set_yticks([])
plt.colorbar(im, ax=axes[0], label='Pearson r')

# Add module boundaries
cumsum = np.cumsum([0] + module_sizes)
for boundary in cumsum[1:-1]:
    axes[0].axhline(boundary - 0.5, color='black', linewidth=1)
    axes[0].axvline(boundary - 0.5, color='black', linewidth=1)

# 2: Module eigengene expression across samples
sample_labels = ['Type1'] * 25 + ['Type2'] * 25
type_order = sorted(range(n_samples), key=lambda i: sample_labels[i])
for i, (mod, eigengene) in enumerate(module_eigengenes.items()):
    axes[1].plot(range(n_samples), eigengene[type_order] + i*3,
                 label=mod.replace('_', ' '), alpha=0.8)
axes[1].axvline(25, color='black', linestyle='--', label='Type1|Type2')
axes[1].set_xlabel('Samples (sorted by type)')
axes[1].set_ylabel('Module eigengene (offset for clarity)')
axes[1].set_title('Module Eigengene Expression')
axes[1].legend(fontsize=7)

# 3: Module-module eigengene correlation
me_df = pd.DataFrame(module_eigengenes)
me_corr = me_df.corr()
im2 = axes[2].imshow(me_corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[2].set_xticks(range(len(module_names)))
axes[2].set_yticks(range(len(module_names)))
axes[2].set_xticklabels([m.replace('_', ' ') for m in module_names], rotation=45, ha='right', fontsize=7)
axes[2].set_yticklabels([m.replace('_', ' ') for m in module_names], fontsize=7)
axes[2].set_title('Module-Module Eigengene Correlation')
plt.colorbar(im2, ax=axes[2])

for i in range(len(module_names)):
    for j in range(len(module_names)):
        axes[2].text(j, i, f'{me_corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()


## 5. Functional Enrichment of Network Modules

After identifying modules, enrichment analysis assigns biological meaning.

### Enrichment methods for networks
1. **Hypergeometric (ORA)**: test if a gene set is overrepresented among module genes
2. **GSEA on eigengene correlations**: rank genes by module membership (kME), run GSEA
3. **Protein complex enrichment**: test overlap with known complexes (CORUM database)

### Multiple testing correction
With many modules × many gene sets:
- Apply **Benjamini-Hochberg FDR** correction
- Report significant at FDR < 0.05 and fold enrichment > 1.5

### Databases for enrichment
- GO Biological Process (gene ontology)
- KEGG pathways
- Reactome pathways
- MSigDB hallmark gene sets
- CORUM protein complexes


In [ ]:
# ----- Functional enrichment per network module -----
from scipy.stats import hypergeom, fisher_exact

# Simulated gene sets (functional annotations)
functional_genesets = {
    'G1/S_transition':      set(['CDK2','CDK4','CCND1','CCNE1','E2F1','RB1','CDKN1A']),
    'Mitosis':              set(['CDK1','CDC25A','E2F3','CDK6','CCND1']),
    'Homologous_recomb':    set(['BRCA1','BRCA2','RAD51','PALB2','FANCD2']),
    'Checkpoint_kinase':    set(['ATM','ATR','CHEK1','CHEK2','BRCA1']),
    'RAS_MAPK_pathway':     set(['KRAS','BRAF','MAP2K1','MAPK1','MAPK3','HRAS','RAF1']),
    'EGFR_signaling':       set(['EGFR','ERBB2','GRB2','SOS1','PIK3CA']),
    'Glycolysis':           set(['LDHA','PKM','HK2','SLC2A1']),
    'Lipid_biosynthesis':   set(['FASN','ACLY','HMGCR','SLC2A1']),
    'T_cell_exhaustion':    set(['PDCD1','CTLA4','LAG3','TIM3','TIGIT','CD8A']),
    'Immune_checkpoint':    set(['CD274','PDCD1','CTLA4','FOXP3','LAG3']),
}

# Compute enrichment for each detected community
background_size = len(all_genes)
enrichment_results = []

for comm_id in range(n_communities):
    comm_genes = set(n for n, c in partition_louvain.items() if c == comm_id)
    n_comm = len(comm_genes)
    
    for gs_name, gs_genes in functional_genesets.items():
        overlap = comm_genes & gs_genes
        k = len(overlap)
        if k == 0:
            continue
        K = len(gs_genes)
        N = background_size
        n = n_comm
        # Hypergeometric test: P(X >= k)
        pval = hypergeom.sf(k-1, N, K, n)
        fold_enr = (k/n) / (K/N) if K > 0 else 0
        enrichment_results.append({
            'Community': f'Comm_{comm_id}',
            'Gene_set': gs_name,
            'Overlap': k,
            'Comm_size': n_comm,
            'GS_size': K,
            'Fold_enrichment': fold_enr,
            'p_value': pval,
        })

df_enr = pd.DataFrame(enrichment_results)
df_enr['-log10p'] = -np.log10(df_enr['p_value'] + 1e-10)
df_enr['Significant'] = df_enr['p_value'] < 0.05

# Pivot for heatmap
df_pivot = df_enr.pivot_table(index='Gene_set', columns='Community',
                               values='-log10p', fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Module Functional Enrichment Analysis', fontsize=12, fontweight='bold')

# Enrichment heatmap
import seaborn as sns
sns.heatmap(df_pivot, ax=axes[0], cmap='YlOrRd', annot=True, fmt='.1f',
            cbar_kws={'label': '-log10(p-value)'})
axes[0].set_title('Module Enrichment Heatmap')
axes[0].tick_params(axis='x', labelsize=8)
axes[0].tick_params(axis='y', labelsize=8)

# Top enriched terms per module
top_enr = df_enr.groupby('Community').apply(lambda x: x.nlargest(3, '-log10p')).reset_index(drop=True)
for comm_id in range(n_communities):
    comm_top = top_enr[top_enr['Community'] == f'Comm_{comm_id}']
    axes[1].barh([f'Comm_{comm_id}: {r["Gene_set"][:20]}' for _, r in comm_top.iterrows()],
                  comm_top['-log10p'], color=det_colors_map[comm_id], alpha=0.8)
axes[1].axvline(-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
axes[1].set_xlabel('-log10(p-value)')
axes[1].set_title('Top 3 Terms per Community')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nSignificant enrichments (p<0.05):")
sig = df_enr[df_enr['Significant']].sort_values('p_value')
print(sig[['Community','Gene_set','Overlap','Fold_enrichment','p_value']].head(10).to_string(index=False))


## 6. Integration with Differential Expression

Overlaying DEG (differentially expressed gene) lists onto network modules reveals **which biological processes are dysregulated** in the condition of interest.

### Workflow
1. Compute DEGs from RNA-seq experiment (edgeR/DESeq2)
2. Map DEGs to network nodes
3. Test enrichment of DEGs per module (Fisher's exact test)
4. Highlight DEG-enriched modules for downstream analysis

### Network-guided interpretation advantage
- A single gene may be significant but not interpretable in isolation
- If it belongs to a module enriched for DEGs, it gains functional context
- Module hub genes among DEGs = high-priority drug targets or biomarkers


In [ ]:
# ----- Overlay DEG results onto network modules -----
from scipy.stats import fisher_exact

# Simulate DEG results (differentially expressed between TumorA vs Normal)
np.random.seed(17)
n_total_genes = len(all_genes)

# Some DEGs are enriched in certain modules
deg_probs = {}
for g in all_genes:
    mod = G.nodes[g]['module_true']
    if mod in ['Cell_cycle', 'MAPK_signaling']:
        deg_probs[g] = 0.6  # more likely to be DE in these modules
    elif mod == 'Immune':
        deg_probs[g] = 0.3
    else:
        deg_probs[g] = 0.15

degs = {g for g in all_genes if np.random.random() < deg_probs[g]}
print(f"Total DEGs: {len(degs)}/{n_total_genes}")

# Test overlap of each module with DEG list
overlap_results = []
for comm_id in range(n_communities):
    comm_genes = set(n for n, c in partition_louvain.items() if c == comm_id)
    overlap = degs & comm_genes
    a = len(overlap)           # DEG in module
    b = len(comm_genes) - a   # not DEG in module
    c = len(degs) - a         # DEG not in module
    d = n_total_genes - len(comm_genes) - c  # not DEG, not in module
    odds, pval = fisher_exact([[a, b], [c, d]], alternative='greater')
    overlap_results.append({
        'Community': f'Comm_{comm_id}',
        'Module_size': len(comm_genes),
        'n_DEG_in_module': a,
        'Odds_ratio': odds,
        'p_value': pval,
        'DEG_fraction': a/len(comm_genes) if comm_genes else 0
    })

df_ov = pd.DataFrame(overlap_results)
df_ov['-log10p'] = -np.log10(df_ov['p_value'] + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('DEG Overlap with Network Modules', fontsize=12, fontweight='bold')

bar_colors_ov = [det_colors_map[i] for i in range(n_communities)]
axes[0].bar(df_ov['Community'], df_ov['DEG_fraction']*100, color=bar_colors_ov, alpha=0.8)
axes[0].axhline(len(degs)/n_total_genes*100, color='black', linestyle='--',
                label=f'Expected ({len(degs)/n_total_genes*100:.0f}%)')
axes[0].set_ylabel('DEG fraction in module (%)')
axes[0].set_title('DEG Enrichment per Module')
axes[0].legend()

axes[1].bar(df_ov['Community'], df_ov['-log10p'], color=bar_colors_ov, alpha=0.8)
axes[1].axhline(-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
axes[1].set_ylabel('-log10(p)')
axes[1].set_title('Fisher Exact: DEG Overlap Significance')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nDEG overlap with modules:")
print(df_ov[['Community','Module_size','n_DEG_in_module','Odds_ratio','p_value']].to_string(index=False))
print(f"\nDEGs in network: {set(all_genes) & degs}")


## 7. Summary

### Key concepts covered

1. **Community detection**: Louvain/Leiden algorithms maximize modularity Q; prefer Leiden for well-connected communities
2. **Modularity Q**: quality metric for partitions; Q > 0.3 = meaningful, Q > 0.5 = strong
3. **WGCNA**: co-expression network modules from expression data; eigengene summarizes module activity
4. **Functional enrichment**: hypergeometric test on each module; FDR correction for multiple testing
5. **DEG integration**: Fisher's exact test for DEG enrichment per module; identifies dysregulated pathways

### Checklist for module analysis
- [ ] Community detection run (Louvain or Leiden)
- [ ] Modularity Q reported
- [ ] Module sizes checked (very small modules may be noise)
- [ ] True labels compared with NMI/ARI if ground truth available
- [ ] GO/KEGG enrichment per module with FDR correction
- [ ] Module eigengene computed for downstream association analysis
- [ ] DEG overlap tested per module

### Next steps
- [Notebook 3: Gene Regulatory Networks](03_gene_regulatory_networks.ipynb) — directed networks of TF regulation


---

[< Previous: PPI Network Construction & Analysis](01_ppi_networks.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Gene Regulatory Network Inference >](03_gene_regulatory_networks.ipynb)

---